In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chat_models.base import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import pandas as pd

load_dotenv()

True

In [34]:
# OpenAI:
# llm = init_chat_model("openai:gpt-3.5-turbo")

# embedding_model = "text-embedding-3-small" # outputs 1536 dimensions by default
# embedding = OpenAIEmbeddings(model=embedding_model)

# Ollama:
llm = ChatOllama(model="gemma3:12b")
embedding = OllamaEmbeddings(model="nomic-embed-text")

In [35]:

fnx_doc = []

try:
    df = pd.read_excel(
    "../DataIngestParsing/data/csv_excel/users_table.xlsx"
    )

    for _, row in df.iterrows():

        text = f"""
        פוליסה מספר {row.get('מספר הפוליסה', '')},
        הפוליסה מנוהלת על ידי {row.get('שם סוכן', '')},
        נוצרה על ידי {row.get('נציג', '')},
        עודכנה על ידי {row.get('עודכן ע"י', '')},
        זוהי פוליסה {"לא פעילה" if row.get("IsActive") == 0 else "פעילה"}.
        """

        # Clean metadata
        metadata = {}

        for key, value in row.items():

            # Skip NaN
            if pd.isna(value):
                continue

            # Convert timestamps to string
            if isinstance(value, pd.Timestamp):
                value = value.isoformat()

            # Convert numpy types to native Python
            if hasattr(value, "item"):
                value = value.item()

            metadata[str(key)] = value

        fnx_doc.append(
            Document(
                page_content=text,
                metadata=metadata
            )
        )

    print(f"The embedding data will look like that: {fnx_doc[0]}")
except Exception as e:
    print(f"Error loading PyPDF: {e}")


The embedding data will look like that: page_content='
        פוליסה מספר 1136442,
        הפוליסה מנוהלת על ידי SMART הפניקס,
        נוצרה על ידי ישי אמסלם,
        עודכנה על ידי שלו קריו,
        זוהי פוליסה פעילה.
        ' metadata={'Id': 29130, 'PolicyId': 250961121136442, 'שנת חיתום': 25, 'BranchNumber': 96, 'Anaf': 112, 'מספר הפוליסה': 1136442, 'מזהה לקוח': 28871, 'תאריך תחילת הפוליסה': '2025-05-17T00:00:00', 'תאריך סיום הפוליסה': '2026-05-11T14:42:00.787000', 'סטטוס הפוליסה': 2, 'StatusDate': '2026-05-11T14:42:00.787000', 'מספר סוכן': 64096, 'שם סוכן': 'SMART הפניקס', 'InstanceId': 7682268845, 'ExternalStatus': 2.0, 'תאריך יצירת הרשומה': '2025-05-17T22:12:56.713000', 'נציג': 'ישי אמסלם', 'CreatorType': 1, 'עודכן ע"י': 'שלו קריו', 'UpdatorType': 2.0, 'UpdateDate': '2026-05-11T14:42:00.787000', 'IsActive': 1, 'IsCancelled': 1.0, 'OriginalPolicyEndDate': '2026-05-31T23:59:59', 'PolicyCancellationReasonCode': 1.0, 'IsAgent': 0, 'UpdatedByUserName': 'usf01266', 'NumberOfDenials': 

In [5]:
 # Vector store: ChromaDB (Chroma.from_documents) automatically infers the dimension from the first embedding it receives — you don't need to configure it manually. It adapts to whatever model you pass.

persist_directory = "./chroma_db_store"
vector_store = Chroma.from_documents(
    documents=fnx_doc,
    embedding=embedding,
    persist_directory=persist_directory, # This will create chroma_db_store folder for the vector store with all the data
    collection_name="rag_collection"
)

print(f"Created vector store with {len(vector_store)} vectors")

Created vector store with 2346 vectors


In [36]:
# Similarity TEST:

query = "תביא לי בבקשה פרטים על פוליסה מספר 1138938"

# similar_docs = vector_store.similarity_search(query, k=2)
similar_docs_with_scores = vector_store.similarity_search_with_score(query, k=2)

# scores with ChromaDB:
# Closer to 0 = Similar result
# Zero means identical vectors

# print(similar_docs[0])
# print("---------")
print(similar_docs_with_scores[0])

(Document(metadata={'CreatorType': 7, 'סטטוס הפוליסה': 2, 'PolicyId': 250961121138938, 'NumberOfClaims': 0.0, 'FnxInsurancePolicyStatus': 1.0, 'תאריך תחילת הפוליסה': '2025-05-25T00:00:00', 'StatusDate': '2026-05-11T12:51:06.100000', 'NumberOfDenials': 0.0, 'UpdatedByUserName': 'usf00271', 'OriginalPolicyEndDate': '2026-05-31T23:59:59', 'ExternalStatus': 2.0, 'UpdatorType': 2.0, 'IsAgent': 0, 'PolicyCancellationReasonCode': 1.0, 'מספר סוכן': 64096, 'InstanceId': 1711462356, 'מזהה לקוח': 29455, 'נציג': 'נתנאל ויצמן', 'תאריך יצירת הרשומה': '2025-05-25T09:05:37.693000', 'מספר הפוליסה': 1138938, 'תאריך סיום הפוליסה': '2026-05-11T12:51:06.100000', 'שנת חיתום': 25, 'IsCancelled': 1.0, 'עודכן ע"י': 'הודיה מכובד', 'IsActive': 1, 'UpdateDate': '2026-05-11T12:51:06.100000', 'BranchNumber': 96, 'Id': 29724, 'שם סוכן': 'SMART הפניקס', 'CreatedByUserName': 'usf89483', 'Anaf': 112}, page_content='\n        פוליסה מספר 1138938,\n        הפוליסה מנוהלת על ידי SMART הפניקס,\n        נוצרה על ידי נתנאל ו

In [37]:
custom_prompt = """
        You are a professional customer service representative for Phoenix Insurance Company.
        ***Always respond in Hebrew regardless of the language used in the question***.

        Your goal is to provide accurate information about the company's policies.

        Guidelines:
        * Use a professional, patient, and respectful tone.
        * If asked analytical questions such as "how many policies...", "what is the last..." etc., respond that you are not an analytics tool, but an information access tool only — DO NOT RETURN ANY DATA!
        * If information is missing or not found in the system — state it clearly.
        * Do not fabricate information that does not exist in the provided data.
        * If multiple similar results are found — ask for additional details to identify the correct one.
        * If the question is unrelated to Phoenix or the data in the system — politely respond that you cannot assist with that topic.
        * Phrase answers in a natural, human way — not as a raw technical data list.

        Example response style:
        * "Your policy is active until 11/05/2026..."
        * "The agent handling your policy is SMART הפניקס..."
        * "No policy was found matching the provided details..."

        ***Always respond in Hebrew***.

        Context: {context}
    """

In [38]:
import re
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import Field
from typing import Any

class HybridRetriever(BaseRetriever):
    vector_store: Any = Field(...)
    k: int = Field(default=3)

    def _get_relevant_documents(self, query: str) -> list[Document]:
        policy_match = re.search(r'\b(\d{6,10})\b', query)

        # 1. The regex + metadata filter is a structured exact lookup
        if policy_match:
            policy_number = int(policy_match.group(1))
            results = self.vector_store.get(
                where={"מספר הפוליסה": policy_number} #  where filter doesn't touch the text at all — it queries the *metadata* dictionary, not page_content.
            )
            docs = [
                Document(page_content=pc, metadata=meta)
                for pc, meta in zip(results["documents"], results["metadatas"])
            ]
            if docs:
                return docs

        # 2. Dense similarity search (semantic search)
        return self.vector_store.similarity_search(query, k=self.k)

retriever = HybridRetriever(vector_store=vector_store)


In [39]:
prompt = ChatPromptTemplate.from_messages([
    ("system", custom_prompt),
    ("human", "{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n        You are a professional customer service representative for Phoenix Insurance Company.\n        ***Always respond in Hebrew regardless of the language used in the question***.\n\n        Your goal is to provide accurate information about the company\'s policies.\n\n        Guidelines:\n        * Use a professional, patient, and respectful tone.\n        * If asked analytical questions such as "how many policies...", "what is the last..." etc., respond that you are not an analytics tool, but an information access tool only — DO NOT RETURN ANY DATA!\n        * If information is missing or not found in the system — state it clearly.\n        * Do not fabricate information that does not exist in the provided data.\n        * If multiple similar results 

In [40]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)


In [42]:
response = rag_chain.invoke({
    "input": "מי מנהל את פוליסה מספר 1014320?",
})

print(response.get("answer"))

שלום רב, 

פוליסה מספר 1014320 מנוהלת על ידי נח סוכנות ביטוח בע"מ. האם יש לכם שאלות נוספות בנוגע לפוליסה זו?
